# Mock Run — Iteration 3 Smoke Test

Exercises all Iteration 3 components with minimal compute:
- **Baselines:** DLinear, PatchTST
- **Core fusion models:** GatedFusion, FiLMFusion, EnsembleFusion (online template descriptions)
- **Fixed architectures:** CrossAttentionFusion (F8), ResidualCorrection (F10, β init=0.1)
- **Stage 1 ablation:** template vs random text source on GatedFusion + FiLMFusion
- **BERTForecaster:** ablation-only, template source
- **Diagnostics:** α (EnsembleFusion), gates (GatedFusion), β (ResidualCorrection), attn (CrossAttentionFusion), cosine sim (3C)
- **Offline embedding mode:** verifies 5-tuple DataLoader when `text_emb_path` is set

`1 epoch · train_fraction=0.05 · seq_len=96 · pred_len=96 · ETTh1`  
No registry writes. Results printed in-notebook only.

## 0. Setup

In [1]:
import os, sys

# Detect project root
_cwd = os.getcwd()
if os.path.isfile(os.path.join(_cwd, 'run_mock.ipynb')):
    PROJECT_ROOT = _cwd
else:
    PROJECT_ROOT = _cwd

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')

Working directory: /Users/egorabrosimov/Projects/multimodality/multimodal_TS_research


In [2]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch

if torch.cuda.is_available():
    device_info = f'CUDA — {torch.cuda.get_device_name(0)}'
elif torch.backends.mps.is_available():
    device_info = 'Apple MPS'
else:
    device_info = 'CPU'

print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device_info}')

PyTorch : 2.10.0
Device  : Apple MPS


## 1. Mock Settings

In [4]:
MOCK_EPOCHS    = 1
MOCK_FRACTION  = 0.05   # 5% of train split
MOCK_SEQ_LEN   = 96     # shorter context window for speed
MOCK_PRED_LEN  = 96
MOCK_DATASET   = 'ETTh1'
MOCK_DATA_PATH = 'ETTh1.csv'
MOCK_ROOT_PATH = './dataset/ETT-small/'

# Iter 3 embedding structure: embeddings/{source}/{dataset}_{split}_minilm.npy
MOCK_EMB_PATH  = 'embeddings/template/ETTh1_train_minilm.npy'
USE_OFFLINE_EMB = os.path.exists(MOCK_EMB_PATH)
print(f'Offline embeddings: {"FOUND — " + MOCK_EMB_PATH if USE_OFFLINE_EMB else "not found — using online encoding"}')

Offline embeddings: not found — using online encoding


## 2. Run Helper

In [5]:
from run_experiment import run, load_config, apply_overrides

mock_results = {}   # label → metrics dict or None

def mock_run(config_path: str, label: str, extra_overrides: list = None):
    """Run one mock experiment and store results."""
    overrides = [
        f'name=mock_{label}',
        f'model.seq_len={MOCK_SEQ_LEN}',
        f'model.pred_len={MOCK_PRED_LEN}',
        f'training.train_epochs={MOCK_EPOCHS}',
        f'training.train_fraction={MOCK_FRACTION}',
        f'training.patience=999',     # disable early stopping
        f'compute.num_workers=0',     # no multiprocessing in notebook
    ]
    if extra_overrides:
        overrides += extra_overrides

    print(f'\n{"="*55}')
    print(f'  {label}')
    print(f'{"="*55}')
    try:
        metrics = run(config_path, overrides=overrides)
        mock_results[label] = metrics
        print(f'  MAE={metrics["mae"]:.4f}  MSE={metrics["mse"]:.4f}')
        if metrics.get('diagnostics'):
            for k, v in metrics['diagnostics'].items():
                print(f'    diag/{k}: {v}')
    except Exception as e:
        print(f'  ERROR: {e}')
        mock_results[label] = None

print('Run helper ready.')

Run helper ready.


## 3. Baseline Models (no text)

In [6]:
BASELINE_CONFIGS = [
    ('experiments/configs/01_dlinear_etth1.yaml',  'dlinear'),
    ('experiments/configs/02_patchtst_etth1.yaml', 'patchtst'),
]

for cfg_path, label in BASELINE_CONFIGS:
    mock_run(cfg_path, label)


  dlinear

Experiment : mock_dlinear
Model      : DLinear
Dataset    : ETTh1
pred_len   : 96
Results in : experiments/results/mock_dlinear_20260413_142001

Using GPU: Apple MPS
Model: DLinear | Total params: 18,624 | Trainable: 18,624


Testing: 100%|██████████| 88/88 [00:00<00:00, 654.80batch/s]


Test | MAE=0.4492  MSE=0.4427  RMSE=0.6653

Done. Results saved to experiments/results/mock_dlinear_20260413_142001
  MAE=0.4492  MSE=0.4427

  patchtst

Experiment : mock_patchtst
Model      : PatchTST
Dataset    : ETTh1
pred_len   : 96
Results in : experiments/results/mock_patchtst_20260413_142027

Using GPU: Apple MPS
Model: PatchTST | Total params: 942,048 | Trainable: 942,048


Testing: 100%|██████████| 22/22 [00:00<00:00, 69.39batch/s]

Test | MAE=0.5655  MSE=0.7117  RMSE=0.8437

Done. Results saved to experiments/results/mock_patchtst_20260413_142027
  MAE=0.5655  MSE=0.7117


## 4. Core Fusion Models — Online Encoding (template)

GatedFusion, FiLMFusion, EnsembleFusion. LateFusion is deprecated and excluded.
Exercises enriched MiniLM-L6 descriptions: regime label, autocorr lag-1/24, temporal context.

In [7]:
TEXT_CONFIGS = [
    ('experiments/configs/05_gated_fusion_etth1.yaml',    'gated_fusion'),
    ('experiments/configs/06_film_fusion_etth1.yaml',     'film_fusion'),
    ('experiments/configs/07_ensemble_fusion_etth1.yaml', 'ensemble_fusion'),
]

for cfg_path, label in TEXT_CONFIGS:
    mock_run(cfg_path, label, extra_overrides=['model.text_source=template'])


  gated_fusion

Experiment : mock_gated_fusion
Model      : GatedFusion
Dataset    : ETTh1
pred_len   : 96
Results in : experiments/results/mock_gated_fusion_20260413_142032

Using GPU: Apple MPS
Model: GatedFusion | Total params: 23,704,544 | Trainable: 991,328


Testing: 100%|██████████| 88/88 [00:02<00:00, 34.67batch/s]


Test | MAE=0.4293  MSE=0.4289  RMSE=0.6549

Done. Results saved to experiments/results/mock_gated_fusion_20260413_142032
  MAE=0.4293  MSE=0.4289

  film_fusion

Experiment : mock_film_fusion
Model      : FiLMFusion
Dataset    : ETTh1
pred_len   : 96
Results in : experiments/results/mock_film_fusion_20260413_142045

Using GPU: Apple MPS
Model: FiLMFusion | Total params: 23,753,824 | Trainable: 1,040,608


Testing: 100%|██████████| 88/88 [00:02<00:00, 33.42batch/s]


Test | MAE=0.4407  MSE=0.4528  RMSE=0.6729

Done. Results saved to experiments/results/mock_film_fusion_20260413_142045
  MAE=0.4407  MSE=0.4528

  ensemble_fusion

Experiment : mock_ensemble_fusion
Model      : EnsembleFusion
Dataset    : ETTh1
pred_len   : 96
Results in : experiments/results/mock_ensemble_fusion_20260413_142052

Using GPU: Apple MPS
Model: EnsembleFusion | Total params: 110,979,073 | Trainable: 1,496,833


Testing: 100%|██████████| 88/88 [00:10<00:00,  8.23batch/s]

Test | MAE=0.4371  MSE=0.4329  RMSE=0.6579

Done. Results saved to experiments/results/mock_ensemble_fusion_20260413_142052
  MAE=0.4371  MSE=0.4329


## 5. Fixed Architectures — F8 CrossAttentionFusion & F10 ResidualCorrection

Stage 0 fixes verified here:
- **F8:** K/V projection produces genuinely distinct tokens (4×d_model linear → reshape)
- **F10:** β init raised 0.01 → **0.1** (fixes gradient starvation at low data fractions)

In [8]:
import yaml
from pathlib import Path

# Build minimal configs inline (no pre-existing YAML needed)
_NEW_MODEL_BASE = {
    'data': {
        'dataset': MOCK_DATASET,
        'root_path': MOCK_ROOT_PATH,
        'data_path': MOCK_DATA_PATH,
        'freq': 'h',
        'features': 'M',
        'target': 'OT',
    },
    'training': {
        'train_epochs': MOCK_EPOCHS,
        'batch_size': 16,
        'learning_rate': 0.0001,
        'patience': 999,
        'seed': 2024,
        'train_fraction': MOCK_FRACTION,
    },
    'compute': {'gpu': 0, 'num_workers': 0, 'use_amp': False},
}

_NEW_MODEL_CFG = {
    'task_name': 'long_term_forecast',
    'enc_in': 7,
    'seq_len': MOCK_SEQ_LEN,
    'label_len': 24,
    'pred_len': MOCK_PRED_LEN,
    'd_model': 64,
    'n_heads': 4,
    'e_layers': 2,
    'd_layers': 1,
    'd_ff': 256,
    'factor': 1,
    'dropout': 0.1,
    'activation': 'gelu',
    'text_model': 'sentence-transformers/all-MiniLM-L6-v2',
    'text_hidden': 384,
    'text_source': 'template',
}

tmp_dir = Path('experiments/configs/mock_tmp')
tmp_dir.mkdir(parents=True, exist_ok=True)

for model_name, label, diag_flags in [
    ('CrossAttentionFusion', 'cross_attn', {'enabled': True, 'log_attn': True}),
    ('ResidualCorrection',   'residual_correction', {'enabled': True, 'log_beta': True}),
]:
    cfg = {
        **_NEW_MODEL_BASE,
        'name': f'mock_{label}',
        'model': {**_NEW_MODEL_CFG, 'name': model_name},
        'diagnostics': diag_flags,
    }
    cfg_path = tmp_dir / f'mock_{label}.yaml'
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
    mock_run(str(cfg_path), label)


  cross_attn

Experiment : mock_cross_attn
Model      : CrossAttentionFusion
Dataset    : ETTh1
pred_len   : 96
Results in : experiments/results/mock_cross_attn_20260413_142118

Using GPU: Apple MPS
Model: CrossAttentionFusion | Total params: 23,003,488 | Trainable: 290,272


Testing: 100%|██████████| 175/175 [00:02<00:00, 62.30batch/s]


Test | MAE=0.5024  MSE=0.5558  RMSE=0.7455

Done. Results saved to experiments/results/mock_cross_attn_20260413_142118
  MAE=0.5024  MSE=0.5558
    diag/attn_mean: 0.25
    diag/attn_per_patch: 0.25

  residual_correction

Experiment : mock_residual_correction
Model      : ResidualCorrection
Dataset    : ETTh1
pred_len   : 96
Results in : experiments/results/mock_residual_correction_20260413_142127

Using GPU: Apple MPS
Model: ResidualCorrection | Total params: 23,091,783 | Trainable: 378,567


Testing: 100%|██████████| 175/175 [00:02<00:00, 65.29batch/s]

Test | MAE=0.4976  MSE=0.5474  RMSE=0.7398

Done. Results saved to experiments/results/mock_residual_correction_20260413_142127
  MAE=0.4976  MSE=0.5474
    diag/beta_mean: 0.10078048706054688
    diag/beta_std: 0.0013827196089550853


## 6. Stage 1 Ablation — Random Text Source

Verifies `text_source=random` returns `torch.randn` embeddings (no semantic content).
Tests GatedFusion + FiLMFusion — the two Stage 1 fusion models.

In [9]:
for model_name, label, diag_flags in [
    ('GatedFusion', 'gated_random',  {'enabled': True, 'log_gates': True}),
    ('FiLMFusion',  'film_random',   {}),
]:
    cfg = {
        **_NEW_MODEL_BASE,
        'name': f'mock_{label}',
        'model': {**_NEW_MODEL_CFG, 'name': model_name, 'text_source': 'random'},
        'diagnostics': diag_flags,
    }
    cfg_path = tmp_dir / f'mock_{label}.yaml'
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
    mock_run(str(cfg_path), label)


  gated_random

Experiment : mock_gated_random
Model      : GatedFusion
Dataset    : ETTh1
pred_len   : 96
Results in : experiments/results/mock_gated_random_20260413_142134

Using GPU: Apple MPS
Model: GatedFusion | Total params: 199,584 | Trainable: 199,584


Testing: 100%|██████████| 175/175 [00:00<00:00, 253.25batch/s]


Test | MAE=0.4995  MSE=0.5603  RMSE=0.7485

Done. Results saved to experiments/results/mock_gated_random_20260413_142134
  MAE=0.4995  MSE=0.5603
    diag/gate_mean: 0.4983502924442291
    diag/gate_std: 0.007615200709551573
    diag/gate_per_dim_mean: [0.5058183670043945, 0.5104625821113586, 0.4875639081001282, 0.5025453567504883, 0.493344247341156, 0.5084352493286133, 0.5033760070800781, 0.4901551902294159, 0.5039846301078796, 0.49581533670425415, 0.5118952989578247, 0.4909360110759735, 0.4897052049636841, 0.4980677664279938, 0.5032389760017395, 0.5033472180366516, 0.505012571811676, 0.5063058733940125, 0.5032046437263489, 0.49745500087738037, 0.4970136880874634, 0.49886003136634827, 0.506853461265564, 0.4958663880825043, 0.49208858609199524, 0.48792701959609985, 0.503485918045044, 0.49140045046806335, 0.5026918649673462, 0.5133128762245178, 0.48939248919487, 0.49221885204315186, 0.4887586236000061, 0.4965631067752838, 0.5112733840942383, 0.5044435858726501, 0.49606043100357056, 0.49

Testing: 100%|██████████| 175/175 [00:00<00:00, 268.75batch/s]

Test | MAE=0.6001  MSE=0.8142  RMSE=0.9023

Done. Results saved to experiments/results/mock_film_random_20260413_142136
  MAE=0.6001  MSE=0.8142


## 7. Offline Embedding Mode (if `.npy` available)

Verifies the 5-tuple DataLoader path when `data.text_emb_path` is set.  
Embeddings live at `embeddings/template/ETTh1_train_minilm.npy` (Iter 3 structure).  
Skip if not yet generated — run §5a of `iteration3_run_experiment.ipynb` first.

In [10]:
if not USE_OFFLINE_EMB:
    print(f'Skipping offline embedding test — {MOCK_EMB_PATH} not found.')
    print('Run §5a of iteration3_run_experiment.ipynb to generate embeddings.')
else:
    for model_name, label in [
        ('GatedFusion',        'gated_offline'),
        ('ResidualCorrection', 'residual_offline'),
    ]:
        cfg = {
            **_NEW_MODEL_BASE,
            'name': f'mock_{label}',
            'model': {**_NEW_MODEL_CFG, 'name': model_name, 'text_source': 'template'},
            'data': {**_NEW_MODEL_BASE['data'], 'text_emb_path': MOCK_EMB_PATH},
        }
        cfg_path = tmp_dir / f'mock_{label}.yaml'
        with open(cfg_path, 'w') as f:
            yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
        mock_run(str(cfg_path), label)

Skipping offline embedding test — embeddings/template/ETTh1_train_minilm.npy not found.
Run §5a of iteration3_run_experiment.ipynb to generate embeddings.


## 8. Diagnostic 3C — Pairwise Cosine Similarity of Text Embeddings

Verifies that `diagnostics.log_cosine_sim` triggers the 3C diagnostic.  
Mean > 0.95 → descriptions are near-identical → text signal is noise.  
Requires offline embeddings (skipped otherwise).

In [11]:
if not USE_OFFLINE_EMB:
    print('Skipping 3C cosine diagnostic — offline embeddings required.')
else:
    cfg = {
        **_NEW_MODEL_BASE,
        'name': 'mock_cosine_diag',
        'model': {**_NEW_MODEL_CFG, 'name': 'GatedFusion', 'text_source': 'template'},
        'data': {**_NEW_MODEL_BASE['data'], 'text_emb_path': MOCK_EMB_PATH},
        'diagnostics': {'enabled': True, 'log_gates': True, 'log_cosine_sim': True},
    }
    cfg_path = tmp_dir / 'mock_cosine_diag.yaml'
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
    mock_run(str(cfg_path), 'cosine_diag')

    m = mock_results.get('cosine_diag')
    if m and m.get('diagnostics', {}).get('cosine_sim_mean') is not None:
        sim = m['diagnostics']['cosine_sim_mean']
        print(f'✓ 3C cosine_sim_mean={sim:.4f}  (>0.95 → descriptions near-identical)')
    else:
        print('✗ cosine_sim_mean missing — check exp_forecasting.py test()')

Skipping 3C cosine diagnostic — offline embeddings required.


## 9. BERTForecaster — Ablation Only (template source)

BERTForecaster is not part of the Stage 2 sweep. Verified here as a text-only ablation baseline.

In [12]:
mock_run(
    'experiments/configs/03_bert_forecaster_etth1.yaml',
    'bert_forecaster',
    extra_overrides=['model.text_source=template'],
)


  bert_forecaster

Experiment : mock_bert_forecaster
Model      : BERTForecaster
Dataset    : ETTh1
pred_len   : 96
Results in : experiments/results/mock_bert_forecaster_20260413_142138

Using GPU: Apple MPS
Model: BERTForecaster | Total params: 22,916,832 | Trainable: 203,616


Testing: 100%|██████████| 88/88 [00:02<00:00, 40.26batch/s]

Test | MAE=0.9494  MSE=1.3366  RMSE=1.1561

Done. Results saved to experiments/results/mock_bert_forecaster_20260413_142138
  MAE=0.9494  MSE=1.3366


## 10. Diagnostic Logging — EnsembleFusion α

Verifies that `diagnostics.log_alpha` causes `diagnostics.alpha_mean/std` to appear in results.

In [13]:
cfg = {
    **_NEW_MODEL_BASE,
    'name': 'mock_ensemble_diag',
    'model': {**_NEW_MODEL_CFG, 'name': 'EnsembleFusion', 'text_source': 'template'},
    'diagnostics': {'enabled': True, 'log_alpha': True},
}
cfg_path = tmp_dir / 'mock_ensemble_diag.yaml'
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

mock_run(str(cfg_path), 'ensemble_diag')

# Verify diagnostic field
m = mock_results.get('ensemble_diag')
if m and m.get('diagnostics'):
    print('✓ diagnostics field present:', list(m['diagnostics'].keys()))
else:
    print('✗ diagnostics field missing — check exp_forecasting.py test() method')


  ensemble_diag

Experiment : mock_ensemble_diag
Model      : EnsembleFusion
Dataset    : ETTh1
pred_len   : 96
Results in : experiments/results/mock_ensemble_diag_20260413_142143

Using GPU: Apple MPS
Model: EnsembleFusion | Total params: 23,092,161 | Trainable: 378,945


Testing: 100%|██████████| 175/175 [00:02<00:00, 61.99batch/s]

Test | MAE=0.4977  MSE=0.5576  RMSE=0.7467

Done. Results saved to experiments/results/mock_ensemble_diag_20260413_142143
  MAE=0.4977  MSE=0.5576
    diag/alpha_mean: 0.4779936248915536
    diag/alpha_std: 0.00036670657326973177
✓ diagnostics field present: ['alpha_mean', 'alpha_std']


## 11. Results Summary

In [14]:
import pandas as pd

rows = []
for label, metrics in mock_results.items():
    if metrics is None:
        rows.append({'model': label, 'MAE': None, 'MSE': None, 'status': 'ERROR'})
    else:
        rows.append({
            'model':  label,
            'MAE':    round(metrics.get('mae', float('nan')), 4),
            'MSE':    round(metrics.get('mse', float('nan')), 4),
            'diags':  ', '.join(metrics.get('diagnostics', {}).keys()) or '—',
            'status': 'OK',
        })

df = pd.DataFrame(rows)
print(f'\n=== Mock Run | {MOCK_DATASET} | pred_len={MOCK_PRED_LEN} | {MOCK_EPOCHS} epoch | fraction={MOCK_FRACTION} ===')
print(df.to_string(index=False))

n_ok  = (df['status'] == 'OK').sum()
n_err = (df['status'] == 'ERROR').sum()
print(f'\n{n_ok} passed, {n_err} failed')


=== Mock Run | ETTh1 | pred_len=96 | 1 epoch | fraction=0.05 ===
              model    MAE    MSE                                  diags status
            dlinear 0.4492 0.4427                                      —     OK
           patchtst 0.5655 0.7117                                      —     OK
       gated_fusion 0.4293 0.4289                                      —     OK
        film_fusion 0.4407 0.4528                                      —     OK
    ensemble_fusion 0.4371 0.4329                                      —     OK
         cross_attn 0.5024 0.5558              attn_mean, attn_per_patch     OK
residual_correction 0.4976 0.5474                    beta_mean, beta_std     OK
       gated_random 0.4995 0.5603 gate_mean, gate_std, gate_per_dim_mean     OK
        film_random 0.6001 0.8142                                      —     OK
    bert_forecaster 0.9494 1.3366                                      —     OK
      ensemble_diag 0.4977 0.5576                  alp

## 12. Cleanup Temp Configs (optional)

In [15]:
import shutil
shutil.rmtree(tmp_dir, ignore_errors=True)
print('Temp configs removed.')

Temp configs removed.
